In [ ]:
!pip install -q pandas requests openpyxl tqdm python-levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.8 MB/s eta 0:00:00


In [ ]:
import getpass

TMDB_KEY = getpass.getpass("TMDb API key (v3): ")
OMDB_KEY = getpass.getpass("OMDb API key: ")

TMDb API key (v3): ··········
OMDb API key: ··········


In [ ]:
from google.colab import files
uploaded = files.upload()           # escolhe o teu Excel (ex.: Livro.xlsx)
FNAME = list(uploaded.keys())[0]
FNAME

Saving Livro.xlsx to Livro (2).xlsx


'Livro (2).xlsx'

In [ ]:
import pandas as pd
import requests, time, re
from tqdm import tqdm
from Levenshtein import ratio as lev_ratio

# ========================= Configs =========================
SHEET_NAME = None          # se None, usa a 1ª sheet automaticamente; ou põe "2024"
SLEEP = 0.25               # pausa pequena para evitar throttle
ALLOW_OMDB_TITLE_FALLBACK = True  # se TMDb falhar, tentar OMDb por título+ano p/ obter imdb_id

# ====================== HTTP helper (retry) =================
def http_get(url, params, retries=3, timeout=20):
    back = 0.6
    for _ in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(back); back *= 2; continue
            return r
        except requests.RequestException:
            time.sleep(back); back *= 2
    return None

# ===================== Normalização de título =====================
_clean_re = re.compile(r"[^a-z0-9 ]+")
def norm_title(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"\s*\(\d{4}\)$", "", s)     # remove "(2023)" no fim
    s = _clean_re.sub(" ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# ========================= TMDb search =========================
def tmdb_search_candidates(title, year):
    """Devolve lista de candidatos TMDb (vários critérios e janelas de ano)."""
    norm = norm_title(title)

    def q(query, y):
        p = {"api_key": TMDB_KEY, "query": query, "include_adult":"false"}
        if y is not None: p["year"] = int(y)
        r = http_get("https://api.themoviedb.org/3/search/movie", p)
        if not r or r.status_code >= 400: return []
        return r.json().get("results", []) or []

    years = [year] if year is not None else [None]
    if year is not None: years += [year-1, year+1]  # janela ±1

    queries = {title, norm}
    cands = []
    seen = set()
    for y in years:
        for qstr in queries:
            for it in q(qstr, y):
                k = it.get("id")
                if k not in seen:
                    seen.add(k); cands.append(it)
    return cands

def composite_score(item, want_title, want_year):
    # sinais
    vc = item.get("vote_count", 0) or 0
    pop = item.get("popularity", 0) or 0.0
    rd  = item.get("release_date") or ""
    yr  = int(rd[:4]) if len(rd)>=4 and rd[:4].isdigit() else None
    y_pen = 0
    if want_year is not None and yr is not None:
        y_pen = abs(yr - want_year)  # quanto mais longe, pior

    # similaridade de título
    sim = lev_ratio(norm_title(item.get("title","")), norm_title(want_title))

    # evitar docs quando possível (genre id 99)
    is_doc = 99 in (item.get("genre_ids") or [])

    # score: grande peso em vote_count, depois popularity, bónus por sim, penalização por ano distante, penaliza doc
    score = (vc * 10) + pop + (sim * 5) - (y_pen * 2) - (5 if is_doc else 0)
    return score, vc, pop, sim, y_pen, is_doc

def pick_best_tmdb(title, year):
    cands = tmdb_search_candidates(title, year)
    if not cands:
        return None
    # se houver qualquer não-documentário, filtramos docs
    non_docs = [x for x in cands if 99 not in (x.get("genre_ids") or [])]
    pool = non_docs if non_docs else cands
    pool.sort(key=lambda x: composite_score(x, title, year), reverse=True)
    return pool[0]

# ========================= TMDb details =========================
def tmdb_details(tmdb_id):
    director = main_actor = imdb_id = tmdb_rating = None

    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}/credits",
                 {"api_key": TMDB_KEY})
    if r and r.status_code < 400:
        d = r.json()
        for p in d.get("crew", []):
            if p.get("job") == "Director":
                director = p.get("name"); break
        cast = d.get("cast", [])
        if cast:
            main_actor = sorted(cast, key=lambda x: x.get("order", 9999))[0].get("name")

    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}/external_ids",
                 {"api_key": TMDB_KEY})
    if r and r.status_code < 400:
        imdb_id = r.json().get("imdb_id")

    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}",
                 {"api_key": TMDB_KEY})
    if r and r.status_code < 400:
        v = r.json().get("vote_average")
        try: tmdb_rating = str(float(v)) if v is not None else None
        except: tmdb_rating = None

    return director, main_actor, imdb_id, tmdb_rating

# ========================= OMDb helpers =========================
def omdb_by_id(imdb_id):
    if not imdb_id: return [None]*6
    r = http_get("https://www.omdbapi.com/", {"i": imdb_id, "apikey": OMDB_KEY})
    if not r or r.status_code >= 400: return [None]*6
    data = r.json()
    if data.get("Response") != "True": return [None]*6
    rotten = None
    for e in data.get("Ratings", []):
        if e.get("Source") == "Rotten Tomatoes":
            rotten = e.get("Value"); break
    return [
        data.get("imdbRating") or None,
        rotten,
        data.get("Rated") or None,
        data.get("Country") or None,
        data.get("Language") or None,
        data.get("Genre") or None,
    ]

def omdb_find_imdb_by_title_year(title, year):
    """Fallback: tenta achar o imdbID pelo OMDb via s=… (search) e y=… (ano)."""
    r = http_get("https://www.omdbapi.com/", {"s": title, "y": year or "", "type":"movie", "apikey": OMDB_KEY})
    if not r or r.status_code >= 400: return None
    data = r.json()
    if data.get("Response") != "True": return None
    # rank simples: título mais parecido + ano mais próximo
    best = None; best_score = -1
    for it in data.get("Search", []):
        t = it.get("Title","")
        y = int(it.get("Year","")[:4]) if it.get("Year","")[:4].isdigit() else None
        sim = lev_ratio(norm_title(t), norm_title(title))
        ypen = abs((year or y or 0) - (y or year or 0)) if (year and y) else 0
        score = (sim*5) - ypen
        if score > best_score:
            best_score = score; best = it
    return best.get("imdbID") if best else None

# ========================= Caches em memória =========================
tmdb_cache = {}   # (title_norm, year) -> (director, actor, imdb_id, tmdb_rating)
omdb_cache = {}   # imdb_id -> [imdb, rotten, rated, country, language, genre]

# ========================= Enrich 1 linha =========================
def enrich_row(title, year, sleep=SLEEP):
    title_norm = norm_title(title)
    key = (title_norm, year)

    # TMDb
    if key in tmdb_cache:
        director, actor, imdb_id, tmdb_rating = tmdb_cache[key]
    else:
        pick = pick_best_tmdb(title, year)
        if not pick and ALLOW_OMDB_TITLE_FALLBACK:
            # última tentativa: usar OMDb search p/ obter imdbID e depois enriquecer via id
            imdb_id = omdb_find_imdb_by_title_year(title, year)
            director = actor = tmdb_rating = None
        elif not pick:
            return [None]*9
        else:
            tmdb_id = pick.get("id")
            director, actor, imdb_id, tmdb_rating = tmdb_details(tmdb_id)
        tmdb_cache[key] = (director, actor, imdb_id, tmdb_rating)
        time.sleep(sleep)

    # OMDb via imdb_id
    if imdb_id in omdb_cache:
        imdb_rating, rotten, rated, country, language, genre = omdb_cache[imdb_id]
    else:
        imdb_rating, rotten, rated, country, language, genre = omdb_by_id(imdb_id)
        omdb_cache[imdb_id] = [imdb_rating, rotten, rated, country, language, genre]
        time.sleep(sleep)

    return [director, actor, tmdb_rating, imdb_rating, rotten, rated, country, language, genre]

# ========================= Pipeline =========================
# Sheet
xls = pd.ExcelFile(FNAME)
if SHEET_NAME is None:
    SHEET_NAME = xls.sheet_names[0]

df = pd.read_excel(FNAME, sheet_name=SHEET_NAME)

# Garantir Year
if "Year" not in df.columns:
    if "Date" in df.columns:
        df["Year"] = pd.to_datetime(df["Date"], errors="coerce").dt.year
    else:
        df["Year"] = None

# Garantir colunas e dtype texto
out_cols = ["Main Director","Main Actor","TMDB Rating","IMDB Rating",
            "Rotten Tomatoes Rating","Rated","Country","Language","Genre"]
for c in out_cols:
    if c not in df.columns: df[c] = None
    df[c] = df[c].astype("object")

# Stats
n_tmdb_ok = n_omdb_fallback = n_failed = 0

for i in tqdm(range(len(df))):
    title = str(df.loc[i, "Title"]).strip()
    year  = df.loc[i, "Year"]
    year  = int(year) if pd.notna(year) else None

    before = tmdb_cache.get((norm_title(title), year))
    res = enrich_row(title, year)

    # contar
    after = tmdb_cache.get((norm_title(title), year))
    if after and after[2]:  # tem imdb_id
        if before is None and ALLOW_OMDB_TITLE_FALLBACK and after[0] is None and after[1] is None:
            n_omdb_fallback += 1
        else:
            n_tmdb_ok += 1
    else:
        n_failed += 1

    df.loc[i, out_cols] = res

OUTFILE = f"enriched_{FNAME}"
df.to_excel(OUTFILE, index=False)
print("✅ Done →", OUTFILE)
print(f"TMDb matched: {n_tmdb_ok} | OMDb fallback: {n_omdb_fallback} | Unmatched: {n_failed}")


100%|██████████| 12426/12426 [08:35<00:00, 24.10it/s]


✅ Done → enriched_Livro (2).xlsx
TMDb matched: 12178 | OMDb fallback: 1 | Unmatched: 247


In [ ]:
from google.colab import files
files.download(OUTFILE)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>